# 1. Import Libraries

In [56]:
import math
from pulp import LpProblem, LpVariable, LpStatus, lpSum, LpMaximize, LpInteger, LpContinuous, value
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import pulp
import random
import warnings
from IPython.display import display
import pandas as pd
import numpy as np
import pulp as pl
from itertools import product

warnings.filterwarnings('ignore')


# 2. Read Data Files 

In [57]:
bom = pd.read_csv('data/bill_of_materials.csv')
cakes = pd.read_csv('data/cakes.csv')
channels = pd.read_csv('data/channels.csv')
ingredients = pd.read_csv('data/ingredients.csv')
demand_params = pd.read_csv('data/instructor_demand_competition.csv')
wages_energy = pd.read_csv('data/wages_energy.csv')
price_table = pd.read_csv('data/price_table_template.csv')

# 3. Extract Wages and Cost Parameters

In [58]:
ingredient_cost = ingredients.set_index('ingredient')['unit_cost_usd']
usage = bom.set_index('cake_id')[ingredients['ingredient'].tolist()].fillna(0)
cake_info = cakes.set_index('cake_id')
channel_info = channels.set_index('channel')
w_params = wages_energy.set_index('parameter')['value']
prep_wage_per_minute = float(w_params['prep_wage_usd_per_hour']) / 60
oven_wage_per_minute = float(w_params['oven_wage_usd_per_hour']) / 60
pack_wage_per_minute = float(w_params['pack_wage_usd_per_hour']) / 60
oven_rental_per_minute = float(w_params['oven_rental_usd_per_hour']) / 60
oven_cost_per_minute = float(w_params['oven_cost_usd_per_hour']) / 60
budget = float(w_params['budget_usd'])

# 4. ILP Solver with P set to price table

In [59]:
# ==== Parameter Preparation (added to fix undefined references) ====
cakes_list = cake_info.index.tolist()
channels_list = channel_info.index.tolist()
service_cap = channel_info['service_cap_per_week']  # channel capacity
transport_cost = channel_info['transport_cost_per_unit_usd']
batch_size = cake_info['batch_size_units']
prep_time_per_unit = cake_info['prep_min_per_unit']
pack_time_per_unit = cake_info['pack_min_per_unit']
pack_cost_per_unit = cake_info['packaging_cost_per_unit_usd']
min_prod = cake_info['minimum_units_if_made']
oven_time_per_batch = cake_info['oven_min_per_batch']


# Ingredient unit cost per cake unit (sum of usage * ingredient cost)
cost_ing = (usage * ingredient_cost).sum(axis=1).to_dict()

# Wages & costs (rename to variables expected later in code)
wage_prep = prep_wage_per_minute * 60  # keep hourly equivalent naming
wage_pack = pack_wage_per_minute * 60
wage_decor = wage_prep  # no decor wage provided; assume same as prep
oven_rental_per_min = oven_rental_per_minute
electricity_per_min = oven_cost_per_minute


# Price pivot (fill NaN with 0 so model runs; user can update price_table.csv)
price_pivot = price_table.pivot(index='cake', columns='channel', values='price').reindex(index=cakes_list, columns=channels_list)
price_pivot = price_pivot.fillna(19.0)


# Demand parameters pivot
alpha_pivot = demand_params.pivot(index='ID', columns='channel', values='alpha').reindex(index=cakes_list, columns=channels_list)
beta_pivot = demand_params.pivot(index='ID', columns='channel', values='beta').reindex(index=cakes_list, columns=channels_list)


# Linear demand D = alpha - beta * price (clipped at >=0)
dem = (alpha_pivot - beta_pivot * price_pivot).clip(lower=0)


# ==== Original Model Code (corrected loops & references) ====
m = pl.LpProblem("Sweet_Market_Simple_ILP", pl.LpMaximize)
# Decision variables
y = pl.LpVariable.dicts("y", (cakes_list, channels_list), lowBound=0, cat=pl.LpInteger)
s = pl.LpVariable.dicts("s", (cakes_list, channels_list), lowBound=0, cat=pl.LpInteger)
b = pl.LpVariable.dicts("b", cakes_list, lowBound=0, cat=pl.LpInteger)


# Demand and sales linkage
for i in cakes_list:
    for j in channels_list:
        Dij = float(dem.loc[i, j])
        m += s[i][j] <= Dij
        m += s[i][j] <= y[i][j]


# Channel service capacity
for j in channels_list:
    m += pl.lpSum(s[i][j] for i in cakes_list) <= float(service_cap.loc[j])


# Batch size & minimum production (if produced)
# Introduce binary to enforce minimum only if produced
z = pl.LpVariable.dicts('z_make', cakes_list, lowBound=0, upBound=1, cat=pl.LpInteger)
M_big = {i: float(dem.loc[i].sum()) for i in cakes_list}  # upper bound on production if made
for i in cakes_list:
    # batch linking
    m += pl.lpSum(y[i][j] for j in channels_list) == int(batch_size.loc[i]) * b[i]
    # minimum if made
    if int(min_prod.loc[i]) > 0:
        m += pl.lpSum(y[i][j] for j in channels_list) - int(min_prod.loc[i]) * z[i] >= 0
        m += pl.lpSum(y[i][j] for j in channels_list) - M_big[i] * z[i] <= 0
    else:
        # if no minimum required, z just equals whether any produced (optional)
        m += pl.lpSum(y[i][j] for j in channels_list) - M_big[i] * z[i] <= 0


# Revenue
revenue = pl.lpSum(float(price_pivot.loc[i, j]) * s[i][j] for i in cakes_list for j in channels_list)


# Costs (match original variable names)
c_ing = pl.lpSum(float(cost_ing.get(i, 0.0)) * y[i][j] for i in cakes_list for j in channels_list)
c_prep = pl.lpSum(float(prep_time_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list) * (wage_prep / 60.0)


# Decor time not provided; assume 0 so cost is 0 (preserve structure)
c_decor = 0 * pl.lpSum(y[i][j] for i in cakes_list for j in channels_list) * (wage_decor / 60.0)
c_pack_labor = pl.lpSum(float(pack_time_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list) * (wage_pack / 60.0)
c_pack_mat = pl.lpSum(float(pack_cost_per_unit.loc[i]) * y[i][j] for i in cakes_list for j in channels_list)
total_oven_minutes = pl.lpSum(float(oven_time_per_batch.loc[i]) * b[i] for i in cakes_list)
c_oven = total_oven_minutes * (oven_rental_per_min + electricity_per_min)
c_transport = pl.lpSum(float(transport_cost.loc[j]) * s[i][j] for i in cakes_list for j in channels_list)
total_cost = c_ing + c_prep + c_decor + c_pack_labor + c_pack_mat + c_oven + c_transport


# Objective
m += revenue - total_cost

# Budget constraint
m += total_cost <= budget

# Solve
_ = m.solve(pl.PULP_CBC_CMD(msg=False))
status = pl.LpStatus[m.status]
profit = pl.value(m.objective)
print("Status:", status)
print("Profit: ${:,.2f}".format(profit))


# Pack results
rows = []
for i in cakes_list:
    for j in channels_list:
        rows.append({
            'cake': i,
            'channel': j,
            'price': float(price_pivot.loc[i, j]),
            'demand_cap': float(dem.loc[i, j]),
            'produced_y': int(pl.value(y[i][j])),
            'sold_s': int(pl.value(s[i][j]))
        })
df_plan = pd.DataFrame(rows)
df_plan.to_csv('plan.csv', index=False)
df_batches = pd.DataFrame({
    'cake': cakes_list,
    'batches': [int(pl.value(b[i])) for i in cakes_list]
})
df_batches.to_csv('batches.csv', index=False)
def v(x): return float(pl.value(x))
breakdown = {
    'Revenue': v(revenue),
    'Ingredients': v(c_ing),
    'Prep labor': v(c_prep),
    'Decor labor': v(c_decor),
    'Packaging labor': v(c_pack_labor),
    'Packaging material': v(c_pack_mat),
    'Oven (rental+energy)': v(c_oven),
    'Transportation': v(c_transport),
    'Total cost': v(total_cost),
    'Profit': v(revenue - total_cost)
}
pd.DataFrame(list(breakdown.items()), columns=['Item', 'Value ($)']).to_csv('costs.csv', index=False)
print('Saved: plan.csv, batches.csv, costs.csv')

Status: Optimal
Profit: $4,230.89
Saved: plan.csv, batches.csv, costs.csv
